# MedMNIST3D suite - 6 datasets x 5 seeds = 30 official runs

Scope of this notebook: **MedMNIST3D only** (Adrenal, Fracture, Nodule, Organ,
Synapse, Vessel), all five registered seeds, one run per physical GPU.

Why this suite first: each 3D dataset has only about 1,000 training volumes, so
30 complete runs are the session target, against roughly 1,160 GPU-hours for
the entire 2D suite. Actual feasibility depends on the first calibration run.

Protocol is unchanged: official source pinned at the recorded commit, official
evaluator, ResNet-50 + 3D, 100 epochs, Adam 0.001 with decays at 50 and 75.


# MedMNIST official baselines runner

Runs the **official** MedMNIST experiment code (pinned commit) on a free cloud GPU
with end-of-epoch checkpointing, so a session disconnect never silently discards a run.

Binding rules:
- The baseline phase uses only the official authors' code plus the documented patches
  written into `LOCAL_PATCHES.md` (seeding + resume; 3D best-checkpoint fix).
- Never fabricate AUC/ACC. A run counts only after its evaluator CSVs and logs
  exist on persistent storage; failed runs stay logged.
- Nothing in this notebook is a result until `status.json` says `verified: true`.

Parallelism policy (protocol-safe):
- `PARALLEL_RUNS` starts **one independent official run per physical GPU**. Each run
  keeps the official batch size, model and 100 epochs, so results stay comparable
  with the published baselines.
- DataParallel / DDP are deliberately NOT used: they would change effective batch
  size and BatchNorm statistics.
- Every run is claimed atomically (`claim_next`), so two GPUs never share a run_id.
- `DATASETS_THIS_SESSION` / `EXCLUDED_RUN_IDS` scope a session so two separate
  sessions can never write the same run directory.
- Training pauses at an epoch boundary before the session cap; the atomic checkpoint
  makes the next session resume instead of restarting from scratch.

How to use (Kaggle):
1. Enable the GPU accelerator (T4 x2 when offered) and internet.
2. Attach exactly one prior state input only when resuming.
3. Start with **Save Version -> Save & Run All**. Kaggle runs the version in a
   fresh session and persists `/kaggle/working` after the notebook exits.
4. For a later version, attach the prior output through one state dataset/input.
   Do not expect Save & Run All to snapshot an already-running interactive session.
5. Rerun this same notebook with the prior state attached; verified runs are skipped
   and `DIMENSIONS_THIS_SESSION` determines whether 2D, 3D, or both are processed.

## 1. Configuration

In [ ]:
# ---- Configuration (fixed for the whole official campaign) ----
UPSTREAM_REPO = "https://github.com/MedMNIST/experiments"
UPSTREAM_COMMIT = "70b6b3a7ad7afddff1df2a3b735235830fbdb142"
EPOCHS = 100
SEEDS = [17, 29, 43, 71, 101]
MAX_SESSION_MINUTES = 660      # hard wall-clock budget; below Kaggle's 12h cap so we
                               # can checkpoint, copy artifacts and save the version
FINALIZE_MARGIN_SECONDS = 900  # extra guard between the pause deadline and the cap
MIN_RUN_MINUTES = 15           # start low: 3D datasets are small (~1k volumes each); recalibrate after the first measured run
                               # at the end of a session
PARALLEL_RUNS = 2              # ONE official run per physical GPU. Never share a GPU:
                               # sharing would change effective batch size / BatchNorm
                               # statistics and break comparability with the official
                               # baselines. This is NOT DataParallel and NOT DDP.
STOP_ON_FAILURE = True         # stop this worker after a non-verified run
DIMENSION = "2d"               # compatibility/default scope for the standard notebooks
DIMENSIONS_THIS_SESSION = ["3d"]
DRIVE_FOLDER = "medmnist_benchmark"
KEEP_BEST_MODEL = True
DATASETS_THIS_SESSION = [
    "adrenalmnist3d",
    "fracturemnist3d",
    "nodulemnist3d",
    "organmnist3d",
    "synapsemnist3d",
    "vesselmnist3d",
]
# 3D suite only: 6 datasets x 5 seeds = 30 runs. Each dataset has only
# ~1000 training volumes, so all 30 runs are expected to fit a single
# session. The 3D epoch time has NOT been measured yet - that is why this
# notebook is prepared first and reviewed before it is launched.
EXCLUDED_RUN_IDS = []

## 2. Persistent storage

In [ ]:
from pathlib import Path
import os, shutil, time
PLATFORM = "kaggle"

BASE = Path('/kaggle/working') / 'medmnist_benchmark'
SESSION_MARKER = Path('/kaggle/working/.medmnist-session-start')
if not SESSION_MARKER.exists():
    SESSION_MARKER.write_text(str(time.time()))
SESSION_START = float(SESSION_MARKER.read_text())

def _looks_like_medmnist_state(candidate):
    """Recognize both old nested exports and Kaggle datasets with files at root."""
    return (candidate.is_dir()
            and (candidate / 'runs').is_dir()
            and ((candidate / 'checkpoints').is_dir() or (candidate / 'state').is_dir()))

previous = set()
for attached in Path('/kaggle/input').iterdir():
    for candidate in (attached, attached / DRIVE_FOLDER):
        if ((candidate.name == DRIVE_FOLDER or candidate == attached)
                and _looks_like_medmnist_state(candidate)):
            previous.add(candidate.resolve())
previous = sorted(previous)
if len(previous) > 1:
    raise RuntimeError('expected zero or one MedMNIST state input; detach stale/duplicate inputs: '
                       + ', '.join(map(str, previous)))
if previous and not (BASE / 'runs').exists():
    shutil.copytree(previous[0], BASE, dirs_exist_ok=True)
    print('restored previous session state from', previous[0])
    print('restored checkpoints:', len(list((BASE / 'checkpoints').glob('*.pt')))
          if (BASE / 'checkpoints').is_dir() else 0)
for sub in ['runs', 'work', 'checkpoints', 'state']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
CODE = Path('/kaggle/working/medmnist-experiments')
print('BASE =', BASE)


## 3. Dependencies (environment is frozen after this cell)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "medmnist==3.0.2", "tensorboardX", "acsconv", "scikit-learn"], check=True)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable a GPU runtime first."
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(BASE / "state" / "pip-freeze.txt").write_text(freeze)
print("environment frozen to", BASE / "state" / "pip-freeze.txt")


## 4. Official source, pinned and verified

In [ ]:
import shutil, subprocess
if CODE.exists() and not (CODE / ".git").exists():
    shutil.rmtree(CODE)
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", UPSTREAM_REPO, str(CODE)], check=True)
subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", UPSTREAM_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
assert head == UPSTREAM_COMMIT, f"unexpected commit: {head}"
print("official source pinned at", head)


## 5. Documented compatibility patches

Seeding, checkpoint/resume, and the 3D best-checkpoint fix. Training math and evaluation are untouched; see `LOCAL_PATCHES.md`.

In [ ]:
# ==== Documented compatibility patches (verbatim from the tested project repo) ====
def patch_3d_checkpoint(source):
    """Return the minimal upstream compatibility patch; refuse unknown input."""
    old = "'net': model.state_dict(),"
    new = "'net': best_model.state_dict(),"
    if source.count(old) != 1 or new in source:
        raise ValueError('Expected exactly one unpatched 3D checkpoint write')
    return source.replace(old, new)

"""Documented compatibility patches for the pinned official MedMNIST experiment scripts.

Reliability and reproducibility only. Training math, model definitions, data
pipelines, optimizer, and evaluation are untouched.

Patches:
1. ``--seed``: deterministic seeding (upstream sets no seed at all).
2. ``--checkpoint`` / ``--resume``: end-of-epoch checkpoint (model, optimizer,
   scheduler, best model, epoch, RNG states, resolved output directory) and
   resume from it. Required because free cloud GPU sessions disconnect;
   without resume a disconnect discards a multi-hour official run.
3. (3D only) save the best-validation model instead of the final-epoch model,
   matching what upstream evaluates (see ``scripts/experiments.py``).

RNG states are restored after initialization. Exact cross-device or cross-version
numerical equivalence is not guaranteed; environment changes must be disclosed.
"""

import importlib.util
from pathlib import Path

SEED_BLOCK = '''
import random


def seed_everything(seed):
    import os
    import numpy as np
    import torch
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
'''

RESUME_BLOCK = '''
    start_epoch = 0
    if resume and checkpoint is not None and os.path.exists(checkpoint):
        _ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
        model.load_state_dict(_ckpt['model'])
        optimizer.load_state_dict(_ckpt['optimizer'])
        scheduler.load_state_dict(_ckpt['scheduler'])
        best_model.load_state_dict(_ckpt['best_model'])
        best_auc = _ckpt['best_auc']
        best_epoch = _ckpt['best_epoch']
        start_epoch = _ckpt['epoch'] + 1
        iteration = _ckpt['iteration']
        random.setstate(_ckpt['python_rng'])
        np.random.set_state(_ckpt['numpy_rng'])
        torch.set_rng_state(_ckpt['torch_rng'])
        if torch.cuda.is_available() and _ckpt['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(_ckpt['cuda_rng'])
        print('==> Resumed from checkpoint at epoch', start_epoch, flush=True)
        del _ckpt
'''

CHECKPOINT_BLOCK = '''
        if checkpoint is not None:
            _ckpt = {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_model': best_model.state_dict(),
                'best_auc': best_auc,
                'best_epoch': best_epoch,
                'epoch': epoch,
                'iteration': iteration,
                'python_rng': random.getstate(),
                'numpy_rng': np.random.get_state(),
                'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                'output_root_resolved': output_root,
            }
            _tmp = checkpoint + '.tmp'
            with open(_tmp, 'wb') as _stream:
                torch.save(_ckpt, _stream)
                _stream.flush()
                os.fsync(_stream.fileno())
            os.replace(_tmp, checkpoint)
            _elapsed = time.time() - _epoch_started
            print('CHECKPOINT completed_epochs=%d epoch_seconds=%.1f' %
                  (epoch + 1, _elapsed), flush=True)
            import json
            _progress = {'completed_epochs': epoch + 1, 'total_epochs': num_epochs,
                         'last_epoch_seconds': _elapsed,
                         'remaining_estimate_hours': (num_epochs - epoch - 1) * _elapsed / 3600}
            with open(checkpoint + '.progress.json.tmp', 'w') as _stream:
                json.dump(_progress, _stream)
            os.replace(checkpoint + '.progress.json.tmp', checkpoint + '.progress.json')
            _deadline = float(os.environ.get('MEDMNIST_DEADLINE', 'inf'))
            if epoch + 1 < num_epochs and time.time() + max(60, _elapsed * 1.5) + 900 >= _deadline:
                print('PAUSED: checkpoint saved; insufficient budget for another epoch', flush=True)
                writer.close()
                return
'''

OUTPUT_ROOT_OLD = '''    output_root = os.path.join(output_root, data_flag, time.strftime("%y%m%d_%H%M%S"))
    if not os.path.exists(output_root):
        os.makedirs(output_root)'''

OUTPUT_ROOT_NEW = '''    if resume and checkpoint is not None and os.path.exists(checkpoint):
        _meta = torch.load(checkpoint, map_location='cpu', weights_only=False)
        output_root = _meta['output_root_resolved']
    else:
        output_root = os.path.join(output_root, data_flag, time.strftime("%y%m%d_%H%M%S"))
    if not os.path.exists(output_root):
        os.makedirs(output_root)'''

ARGPARSE_OLD = "    parser.add_argument('--run',"
ARGPARSE_NEW = '''    parser.add_argument('--seed', default=None, type=int)
    parser.add_argument('--checkpoint', default=None, type=str)
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--run','''

_SIGNATURES = {
    '2d': "def main(data_flag, output_root, num_epochs, gpu_ids, batch_size, size, download, model_flag, resize, as_rgb, model_path, run):",
    '3d': "def main(data_flag, output_root, num_epochs, gpu_ids, batch_size, size, conv, pretrained_3d, download, model_flag, as_rgb, shape_transform, model_path, run):",
}

_CALLS = {
    '2d': "main(data_flag, output_root, num_epochs, gpu_ids, batch_size, size, download, model_flag, resize, as_rgb, model_path, run)",
    '3d': "main(data_flag, output_root, num_epochs, gpu_ids, batch_size, size, conv, pretrained_3d, download, model_flag, as_rgb, shape_transform, model_path, run)",
}

LOCAL_PATCHES_MD = """# Local compatibility patches

Official code pinned from https://github.com/MedMNIST/experiments
(commit 70b6b3a7ad7afddff1df2a3b735235830fbdb142). Training math, model
definitions, data pipelines, optimizer, and evaluation are untouched.

1. `--seed`: deterministic seeding; upstream sets no seed.
2. `--checkpoint` / `--resume`: end-of-epoch checkpoint and resume so a
   disconnected free-GPU session does not discard a multi-hour official run.
   RNG states are restored on the CPU after initialization. Cross-device and
   cross-version bitwise reproducibility is not guaranteed. Epoch checkpoints
   use flush/fsync and atomic replacement. A session deadline pauses at an
   epoch boundary; a launcher watchdog handles a stalled epoch.
3. (3D only) `best_model.pth` now stores the best-validation model that
   upstream evaluates; upstream saved the final-epoch model under that name.
4. Respect the launcher's GPU visibility so independent seeds use separate
   GPUs with the original batch size, not DataParallel or distributed training.
"""


def _load_3d_checkpoint_patch():
    existing = globals().get('patch_3d_checkpoint')
    if existing is not None:
        return existing
    try:
        from experiments import patch_3d_checkpoint as fn
        return fn
    except ImportError:
        spec = importlib.util.spec_from_file_location(
            'experiments', Path(__file__).with_name('experiments.py'))
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module.patch_3d_checkpoint


def _replace_once(source, old, new, label):
    if source.count(old) != 1:
        raise ValueError(f'patch anchor must appear exactly once: {label}')
    return source.replace(old, new, 1)


def patch_source(source, dimension):
    """Apply the documented patches to an official training script source."""
    if dimension not in ('2d', '3d'):
        raise ValueError("dimension must be '2d' or '3d'")
    if 'start_epoch' in source or '--seed' in source:
        raise ValueError('source is already patched')
    if dimension == '3d':
        source = _load_3d_checkpoint_patch()(source)

    source = _replace_once(
        source, 'from tqdm import trange', 'from tqdm import trange' + SEED_BLOCK, 'trange import')

    signature = _SIGNATURES[dimension]
    source = _replace_once(
        source, signature,
        signature[:-2] + ', seed=None, checkpoint=None, resume=False):'
        + '\n    if seed is not None:\n        seed_everything(seed)\n', 'main signature')

    source = _replace_once(source, OUTPUT_ROOT_OLD, OUTPUT_ROOT_NEW, 'output root')
    source = _replace_once(
        source, 'os.environ["CUDA_VISIBLE_DEVICES"]=str(gpu_ids[0])',
        'os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(gpu_ids[0]))', 'GPU visibility')

    source = _replace_once(
        source, '    global iteration\n    iteration = 0',
        '    global iteration\n    iteration = 0\n' + RESUME_BLOCK, 'resume block')

    source = _replace_once(
        source, '    for epoch in trange(num_epochs):',
        '    for epoch in trange(start_epoch, num_epochs):\n        _epoch_started = time.time()', 'loop start')

    source = _replace_once(
        source, "            print('cur_best_epoch', best_epoch)\n",
        "            print('cur_best_epoch', best_epoch)\n" + CHECKPOINT_BLOCK, 'checkpoint save')

    source = _replace_once(source, ARGPARSE_OLD, ARGPARSE_NEW, 'argparse args')

    call = _CALLS[dimension]
    source = _replace_once(
        source, call,
        call[:-1] + ', seed=args.seed, checkpoint=args.checkpoint, resume=args.resume)',
        'call site')
    return source

# ==== Tested reliability helpers (verbatim from scripts/runner_safety.py) ====
import hashlib
import json
import os
from pathlib import Path
import re
import subprocess
import time


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def child_environment(gpu_index, seed, deadline):
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES=str(gpu_index), PYTHONHASHSEED=str(seed),
               MEDMNIST_DEADLINE=str(deadline), PYTHONUNBUFFERED='1',
               OMP_NUM_THREADS='2', MKL_NUM_THREADS='2',
               CUBLAS_WORKSPACE_CONFIG=':4096:8')
    return env


def select_session_runs(queue, dimension, datasets, excluded, is_done, limit):
    selected = []
    seen = set(excluded)
    for run in queue:
        if (run['dimension'] != dimension or run['dataset'] not in datasets
                or run['run_id'] in seen or is_done(run)):
            continue
        selected.append(run)
        seen.add(run['run_id'])
        if len(selected) >= limit:
            break
    return selected


def run_process(command, cwd, env, log, deadline, poll_seconds=30):
    """Hard backstop; normal pause happens after an atomic epoch checkpoint."""
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=log,
                               stderr=subprocess.STDOUT)
    timed_out = False
    try:
        while process.poll() is None:
            remaining = deadline - time.time()
            if remaining <= 0:
                timed_out = True
                process.terminate()
                break
            try:
                process.wait(timeout=min(poll_seconds, remaining))
            except subprocess.TimeoutExpired:
                pass
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
        process.wait()
    return process.returncode, timed_out


def is_resumable_pause(exit_code, timed_out, completed_epochs, expected_epochs):
    """True only when a real epoch checkpoint can safely continue later."""
    return (0 < completed_epochs < expected_epochs
            and (timed_out or exit_code == 0))


def checkpoint_completed_epochs(state):
    """The atomic checkpoint is authoritative; progress JSON is informational."""
    epoch = state.get('epoch')
    if not isinstance(epoch, int) or epoch < 0:
        raise ValueError('checkpoint has no valid completed epoch')
    return epoch + 1


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def verify_artifacts(run, resolved, completed_epochs, expected_epochs,
                     evaluator_factory=None):
    """Recompute all split metrics with official labels, without rewriting CSVs."""
    import csv
    import numpy as np
    if evaluator_factory is None:
        from medmnist import Evaluator
        evaluator_factory = Evaluator
    if completed_epochs != expected_epochs:
        raise ValueError('training epochs incomplete or incompatible')
    resolved = Path(resolved)
    log_path = resolved / (run['dataset'] + '_log.txt')
    model_path = resolved / 'best_model.pth'
    if not log_path.is_file():
        raise ValueError('final evaluator log is missing')
    if not model_path.is_file() or model_path.stat().st_size == 0:
        raise ValueError('best-validation model is missing')
    logs = log_path.read_text(encoding='utf-8')
    hashes = {p.name: file_sha256(p) for p in (log_path, model_path)}
    verified = {}
    for split in ('train', 'val', 'test'):
        files = list(resolved.glob(f"{run['dataset']}_{split}_*@{run['run_id']}.csv"))
        if len(files) != 1:
            raise ValueError(f'{split}: expected exactly one prediction CSV')
        with files[0].open(newline='', encoding='utf-8') as stream:
            rows = [row for row in csv.reader(stream) if row]
        table = np.asarray([[float(cell) for cell in row] for row in rows])
        index = table[:, 0].astype(np.int64)
        scores = table[:, 1:]
        evaluator = evaluator_factory(run['dataset'], split, size=28)
        n = len(evaluator.labels)
        if len(scores) != n or not np.array_equal(index, np.arange(n)):
            raise ValueError(f'{split}: prediction rows or ordering disagree with official labels')
        expected_columns = len(evaluator.info['label'])
        if scores.shape != (n, expected_columns):
            raise ValueError(f'{split}: incorrect prediction shape')
        if not np.isfinite(scores).all() or (scores < 0).any() or (scores > 1).any():
            raise ValueError(f'{split}: invalid prediction probabilities')
        auc, acc = evaluator.evaluate(scores)
        matches = re.findall(rf'{split}  auc: ([0-9.]+)  acc: ([0-9.]+)', logs)
        if (not matches or not np.isfinite([auc, acc]).all()
                or not np.allclose([auc, acc], [float(x) for x in matches[-1]],
                                   rtol=0, atol=0.0000051)):
            raise ValueError(f'{split}: recomputed metrics disagree with final log')
        expected_name = f"{run['dataset']}_{split}_[AUC]{auc:.3f}_[ACC]{acc:.3f}@{run['run_id']}.csv"
        if files[0].name != expected_name:
            raise ValueError(f'{split}: recomputed metrics disagree with CSV name')
        verified[split + '_auc'], verified[split + '_acc'] = float(auc), float(acc)
        hashes[files[0].name] = file_sha256(files[0])
    return dict(verified, artifact_sha256=hashes,
                verification_method='official-evaluator-recomputed-from-predictions')

for _dimension in DIMENSIONS_THIS_SESSION:
    DIM_DIR = "MedMNIST2D" if _dimension == "2d" else "MedMNIST3D"
    TARGET = CODE / DIM_DIR / "train_and_eval_pytorch.py"
    _relative = f"{DIM_DIR}/train_and_eval_pytorch.py"
    _src = subprocess.check_output(["git", "-C", str(CODE), "show",
                                    f"{UPSTREAM_COMMIT}:{_relative}"], text=True)
    TARGET.write_text(patch_source(_src, _dimension), encoding="utf-8")
    compile(TARGET.read_text(encoding="utf-8"), str(TARGET), "exec")
    print("patched official script:", TARGET)
(CODE / "LOCAL_PATCHES.md").write_text(LOCAL_PATCHES_MD, encoding="utf-8")


## 6. Preregistered run queue

In [ ]:
import json, re, time
PLAN_RUNS = json.loads(r'''[
 {
  "run_id": "2d-chestmnist-resnet50-224-seed17-r1",
  "dataset": "chestmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.773,
  "reported_acc": 0.948,
  "prior_attempts": "Three prior local attempts failed before completing one epoch. Only the retained cloud record is published in this repository."
 },
 {
  "run_id": "2d-chestmnist-resnet50-224-seed29-r1",
  "dataset": "chestmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.773,
  "reported_acc": 0.948
 },
 {
  "run_id": "2d-chestmnist-resnet50-224-seed43-r1",
  "dataset": "chestmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.773,
  "reported_acc": 0.948
 },
 {
  "run_id": "2d-chestmnist-resnet50-224-seed71-r1",
  "dataset": "chestmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.773,
  "reported_acc": 0.948
 },
 {
  "run_id": "2d-chestmnist-resnet50-224-seed101-r1",
  "dataset": "chestmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.773,
  "reported_acc": 0.948
 },
 {
  "run_id": "2d-bloodmnist-resnet50-224-seed17-r1",
  "dataset": "bloodmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.997,
  "reported_acc": 0.95
 },
 {
  "run_id": "2d-bloodmnist-resnet50-224-seed29-r1",
  "dataset": "bloodmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.997,
  "reported_acc": 0.95
 },
 {
  "run_id": "2d-bloodmnist-resnet50-224-seed43-r1",
  "dataset": "bloodmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.997,
  "reported_acc": 0.95
 },
 {
  "run_id": "2d-bloodmnist-resnet50-224-seed71-r1",
  "dataset": "bloodmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.997,
  "reported_acc": 0.95
 },
 {
  "run_id": "2d-bloodmnist-resnet50-224-seed101-r1",
  "dataset": "bloodmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.997,
  "reported_acc": 0.95
 },
 {
  "run_id": "2d-breastmnist-resnet50-224-seed17-r1",
  "dataset": "breastmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.866,
  "reported_acc": 0.842
 },
 {
  "run_id": "2d-breastmnist-resnet50-224-seed29-r1",
  "dataset": "breastmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.866,
  "reported_acc": 0.842
 },
 {
  "run_id": "2d-breastmnist-resnet50-224-seed43-r1",
  "dataset": "breastmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.866,
  "reported_acc": 0.842
 },
 {
  "run_id": "2d-breastmnist-resnet50-224-seed71-r1",
  "dataset": "breastmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.866,
  "reported_acc": 0.842
 },
 {
  "run_id": "2d-breastmnist-resnet50-224-seed101-r1",
  "dataset": "breastmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.866,
  "reported_acc": 0.842
 },
 {
  "run_id": "2d-dermamnist-resnet50-224-seed17-r1",
  "dataset": "dermamnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.912,
  "reported_acc": 0.731
 },
 {
  "run_id": "2d-dermamnist-resnet50-224-seed29-r1",
  "dataset": "dermamnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.912,
  "reported_acc": 0.731
 },
 {
  "run_id": "2d-dermamnist-resnet50-224-seed43-r1",
  "dataset": "dermamnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.912,
  "reported_acc": 0.731
 },
 {
  "run_id": "2d-dermamnist-resnet50-224-seed71-r1",
  "dataset": "dermamnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.912,
  "reported_acc": 0.731
 },
 {
  "run_id": "2d-dermamnist-resnet50-224-seed101-r1",
  "dataset": "dermamnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.912,
  "reported_acc": 0.731
 },
 {
  "run_id": "2d-octmnist-resnet50-224-seed17-r1",
  "dataset": "octmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.958,
  "reported_acc": 0.776
 },
 {
  "run_id": "2d-octmnist-resnet50-224-seed29-r1",
  "dataset": "octmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.958,
  "reported_acc": 0.776
 },
 {
  "run_id": "2d-octmnist-resnet50-224-seed43-r1",
  "dataset": "octmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.958,
  "reported_acc": 0.776
 },
 {
  "run_id": "2d-octmnist-resnet50-224-seed71-r1",
  "dataset": "octmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.958,
  "reported_acc": 0.776
 },
 {
  "run_id": "2d-octmnist-resnet50-224-seed101-r1",
  "dataset": "octmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.958,
  "reported_acc": 0.776
 },
 {
  "run_id": "2d-organamnist-resnet50-224-seed17-r1",
  "dataset": "organamnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.998,
  "reported_acc": 0.947
 },
 {
  "run_id": "2d-organamnist-resnet50-224-seed29-r1",
  "dataset": "organamnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.998,
  "reported_acc": 0.947
 },
 {
  "run_id": "2d-organamnist-resnet50-224-seed43-r1",
  "dataset": "organamnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.998,
  "reported_acc": 0.947
 },
 {
  "run_id": "2d-organamnist-resnet50-224-seed71-r1",
  "dataset": "organamnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.998,
  "reported_acc": 0.947
 },
 {
  "run_id": "2d-organamnist-resnet50-224-seed101-r1",
  "dataset": "organamnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.998,
  "reported_acc": 0.947
 },
 {
  "run_id": "2d-organcmnist-resnet50-224-seed17-r1",
  "dataset": "organcmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.993,
  "reported_acc": 0.911
 },
 {
  "run_id": "2d-organcmnist-resnet50-224-seed29-r1",
  "dataset": "organcmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.993,
  "reported_acc": 0.911
 },
 {
  "run_id": "2d-organcmnist-resnet50-224-seed43-r1",
  "dataset": "organcmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.993,
  "reported_acc": 0.911
 },
 {
  "run_id": "2d-organcmnist-resnet50-224-seed71-r1",
  "dataset": "organcmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.993,
  "reported_acc": 0.911
 },
 {
  "run_id": "2d-organcmnist-resnet50-224-seed101-r1",
  "dataset": "organcmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.993,
  "reported_acc": 0.911
 },
 {
  "run_id": "2d-organsmnist-resnet50-224-seed17-r1",
  "dataset": "organsmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.975,
  "reported_acc": 0.785
 },
 {
  "run_id": "2d-organsmnist-resnet50-224-seed29-r1",
  "dataset": "organsmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.975,
  "reported_acc": 0.785
 },
 {
  "run_id": "2d-organsmnist-resnet50-224-seed43-r1",
  "dataset": "organsmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.975,
  "reported_acc": 0.785
 },
 {
  "run_id": "2d-organsmnist-resnet50-224-seed71-r1",
  "dataset": "organsmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.975,
  "reported_acc": 0.785
 },
 {
  "run_id": "2d-organsmnist-resnet50-224-seed101-r1",
  "dataset": "organsmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.975,
  "reported_acc": 0.785
 },
 {
  "run_id": "2d-pathmnist-resnet50-224-seed17-r1",
  "dataset": "pathmnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.989,
  "reported_acc": 0.892
 },
 {
  "run_id": "2d-pathmnist-resnet50-224-seed29-r1",
  "dataset": "pathmnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.989,
  "reported_acc": 0.892
 },
 {
  "run_id": "2d-pathmnist-resnet50-224-seed43-r1",
  "dataset": "pathmnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.989,
  "reported_acc": 0.892
 },
 {
  "run_id": "2d-pathmnist-resnet50-224-seed71-r1",
  "dataset": "pathmnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.989,
  "reported_acc": 0.892
 },
 {
  "run_id": "2d-pathmnist-resnet50-224-seed101-r1",
  "dataset": "pathmnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.989,
  "reported_acc": 0.892
 },
 {
  "run_id": "2d-pneumoniamnist-resnet50-224-seed17-r1",
  "dataset": "pneumoniamnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.962,
  "reported_acc": 0.884
 },
 {
  "run_id": "2d-pneumoniamnist-resnet50-224-seed29-r1",
  "dataset": "pneumoniamnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.962,
  "reported_acc": 0.884
 },
 {
  "run_id": "2d-pneumoniamnist-resnet50-224-seed43-r1",
  "dataset": "pneumoniamnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.962,
  "reported_acc": 0.884
 },
 {
  "run_id": "2d-pneumoniamnist-resnet50-224-seed71-r1",
  "dataset": "pneumoniamnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.962,
  "reported_acc": 0.884
 },
 {
  "run_id": "2d-pneumoniamnist-resnet50-224-seed101-r1",
  "dataset": "pneumoniamnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.962,
  "reported_acc": 0.884
 },
 {
  "run_id": "2d-retinamnist-resnet50-224-seed17-r1",
  "dataset": "retinamnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.716,
  "reported_acc": 0.511
 },
 {
  "run_id": "2d-retinamnist-resnet50-224-seed29-r1",
  "dataset": "retinamnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.716,
  "reported_acc": 0.511
 },
 {
  "run_id": "2d-retinamnist-resnet50-224-seed43-r1",
  "dataset": "retinamnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.716,
  "reported_acc": 0.511
 },
 {
  "run_id": "2d-retinamnist-resnet50-224-seed71-r1",
  "dataset": "retinamnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.716,
  "reported_acc": 0.511
 },
 {
  "run_id": "2d-retinamnist-resnet50-224-seed101-r1",
  "dataset": "retinamnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.716,
  "reported_acc": 0.511
 },
 {
  "run_id": "2d-tissuemnist-resnet50-224-seed17-r1",
  "dataset": "tissuemnist",
  "seed": 17,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.932,
  "reported_acc": 0.68
 },
 {
  "run_id": "2d-tissuemnist-resnet50-224-seed29-r1",
  "dataset": "tissuemnist",
  "seed": 29,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.932,
  "reported_acc": 0.68
 },
 {
  "run_id": "2d-tissuemnist-resnet50-224-seed43-r1",
  "dataset": "tissuemnist",
  "seed": 43,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.932,
  "reported_acc": 0.68
 },
 {
  "run_id": "2d-tissuemnist-resnet50-224-seed71-r1",
  "dataset": "tissuemnist",
  "seed": 71,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.932,
  "reported_acc": 0.68
 },
 {
  "run_id": "2d-tissuemnist-resnet50-224-seed101-r1",
  "dataset": "tissuemnist",
  "seed": 101,
  "dimension": "2d",
  "resolution": 224,
  "batch_size": 128,
  "reported_auc": 0.932,
  "reported_acc": 0.68
 },
 {
  "run_id": "3d-adrenalmnist3d-resnet50-28-seed17-r1",
  "dataset": "adrenalmnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.828,
  "reported_acc": 0.745
 },
 {
  "run_id": "3d-adrenalmnist3d-resnet50-28-seed29-r1",
  "dataset": "adrenalmnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.828,
  "reported_acc": 0.745
 },
 {
  "run_id": "3d-adrenalmnist3d-resnet50-28-seed43-r1",
  "dataset": "adrenalmnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.828,
  "reported_acc": 0.745
 },
 {
  "run_id": "3d-adrenalmnist3d-resnet50-28-seed71-r1",
  "dataset": "adrenalmnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.828,
  "reported_acc": 0.745
 },
 {
  "run_id": "3d-adrenalmnist3d-resnet50-28-seed101-r1",
  "dataset": "adrenalmnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.828,
  "reported_acc": 0.745
 },
 {
  "run_id": "3d-fracturemnist3d-resnet50-28-seed17-r1",
  "dataset": "fracturemnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.725,
  "reported_acc": 0.494
 },
 {
  "run_id": "3d-fracturemnist3d-resnet50-28-seed29-r1",
  "dataset": "fracturemnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.725,
  "reported_acc": 0.494
 },
 {
  "run_id": "3d-fracturemnist3d-resnet50-28-seed43-r1",
  "dataset": "fracturemnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.725,
  "reported_acc": 0.494
 },
 {
  "run_id": "3d-fracturemnist3d-resnet50-28-seed71-r1",
  "dataset": "fracturemnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.725,
  "reported_acc": 0.494
 },
 {
  "run_id": "3d-fracturemnist3d-resnet50-28-seed101-r1",
  "dataset": "fracturemnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.725,
  "reported_acc": 0.494
 },
 {
  "run_id": "3d-nodulemnist3d-resnet50-28-seed17-r1",
  "dataset": "nodulemnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.875,
  "reported_acc": 0.847
 },
 {
  "run_id": "3d-nodulemnist3d-resnet50-28-seed29-r1",
  "dataset": "nodulemnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.875,
  "reported_acc": 0.847
 },
 {
  "run_id": "3d-nodulemnist3d-resnet50-28-seed43-r1",
  "dataset": "nodulemnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.875,
  "reported_acc": 0.847
 },
 {
  "run_id": "3d-nodulemnist3d-resnet50-28-seed71-r1",
  "dataset": "nodulemnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.875,
  "reported_acc": 0.847
 },
 {
  "run_id": "3d-nodulemnist3d-resnet50-28-seed101-r1",
  "dataset": "nodulemnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.875,
  "reported_acc": 0.847
 },
 {
  "run_id": "3d-organmnist3d-resnet50-28-seed17-r1",
  "dataset": "organmnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.994,
  "reported_acc": 0.883
 },
 {
  "run_id": "3d-organmnist3d-resnet50-28-seed29-r1",
  "dataset": "organmnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.994,
  "reported_acc": 0.883
 },
 {
  "run_id": "3d-organmnist3d-resnet50-28-seed43-r1",
  "dataset": "organmnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.994,
  "reported_acc": 0.883
 },
 {
  "run_id": "3d-organmnist3d-resnet50-28-seed71-r1",
  "dataset": "organmnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.994,
  "reported_acc": 0.883
 },
 {
  "run_id": "3d-organmnist3d-resnet50-28-seed101-r1",
  "dataset": "organmnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.994,
  "reported_acc": 0.883
 },
 {
  "run_id": "3d-synapsemnist3d-resnet50-28-seed17-r1",
  "dataset": "synapsemnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.851,
  "reported_acc": 0.795
 },
 {
  "run_id": "3d-synapsemnist3d-resnet50-28-seed29-r1",
  "dataset": "synapsemnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.851,
  "reported_acc": 0.795
 },
 {
  "run_id": "3d-synapsemnist3d-resnet50-28-seed43-r1",
  "dataset": "synapsemnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.851,
  "reported_acc": 0.795
 },
 {
  "run_id": "3d-synapsemnist3d-resnet50-28-seed71-r1",
  "dataset": "synapsemnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.851,
  "reported_acc": 0.795
 },
 {
  "run_id": "3d-synapsemnist3d-resnet50-28-seed101-r1",
  "dataset": "synapsemnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.851,
  "reported_acc": 0.795
 },
 {
  "run_id": "3d-vesselmnist3d-resnet50-28-seed17-r1",
  "dataset": "vesselmnist3d",
  "seed": 17,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.907,
  "reported_acc": 0.918
 },
 {
  "run_id": "3d-vesselmnist3d-resnet50-28-seed29-r1",
  "dataset": "vesselmnist3d",
  "seed": 29,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.907,
  "reported_acc": 0.918
 },
 {
  "run_id": "3d-vesselmnist3d-resnet50-28-seed43-r1",
  "dataset": "vesselmnist3d",
  "seed": 43,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.907,
  "reported_acc": 0.918
 },
 {
  "run_id": "3d-vesselmnist3d-resnet50-28-seed71-r1",
  "dataset": "vesselmnist3d",
  "seed": 71,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.907,
  "reported_acc": 0.918
 },
 {
  "run_id": "3d-vesselmnist3d-resnet50-28-seed101-r1",
  "dataset": "vesselmnist3d",
  "seed": 101,
  "dimension": "3d",
  "resolution": 28,
  "batch_size": 32,
  "reported_auc": 0.907,
  "reported_acc": 0.918
 }
]''')

def run_dir(run):
    return BASE / "runs" / run["run_id"]

def read_status(run):
    p = run_dir(run) / "status.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

def is_done(run):
    st = read_status(run)
    return st is not None and st.get("status") == "completed" and st.get("verified") is True

def next_pending(dimension):
    for run in PLAN_RUNS:
        if run["dimension"] == dimension and not is_done(run):
            return run
    return None

pending = [r for r in PLAN_RUNS
           if r["dataset"] in DATASETS_THIS_SESSION and not is_done(r)]
print(f"{len(pending)} pending runs for this session; next:",
      pending[0]["run_id"] if pending else "none")


## 7. Launcher (resume-safe, artifact-verified)

In [ ]:
import torch
import threading
import medmnist

# One official run per physical GPU. The child process only ever sees its own GPU
# through CUDA_VISIBLE_DEVICES (set by child_environment), so --gpu_ids stays 0 and
# the official batch size / BatchNorm statistics are untouched.
LAUNCHER_LOCK = threading.Lock()
DOWNLOAD_LOCK = threading.Lock()

def ensure_dataset_downloaded(run):
    """Avoid two workers downloading/writing the same MedMNIST NPZ at once."""
    with DOWNLOAD_LOCK:
        info = medmnist.INFO[run["dataset"]]
        DataClass = getattr(medmnist, info["python_class"])
        DataClass(split="train", download=True, size=28)
        print("dataset ready:", run["dataset"])

def session_deadline():
    """Wall-clock instant at which training must pause at an epoch boundary."""
    return SESSION_START + MAX_SESSION_MINUTES * 60 - FINALIZE_MARGIN_SECONDS

def read_progress(ckpt):
    p = Path(str(ckpt) + ".progress.json")
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

def launch(run, gpu_index=0):
    rd = run_dir(run)
    rd.mkdir(parents=True, exist_ok=True)
    ckpt = BASE / "checkpoints" / (run["run_id"] + ".pt")
    dim_dir = "MedMNIST2D" if run["dimension"] == "2d" else "MedMNIST3D"
    script = CODE / dim_dir / "train_and_eval_pytorch.py"
    out_root = BASE / "work" / "output" / run["run_id"]
    deadline = session_deadline()
    cmd = [sys.executable, str(script),
           "--data_flag", run["dataset"], "--output_root", str(out_root),
           "--num_epochs", str(EPOCHS), "--size", "28", "--gpu_ids", "0",
           "--batch_size", str(run["batch_size"]), "--model_flag", "resnet50",
           "--run", run["run_id"], "--seed", str(run["seed"]),
           "--checkpoint", str(ckpt), "--resume", "--download"]
    if run["dimension"] == "2d":
        cmd += ["--resize", "--as_rgb"]
    else:
        cmd += ["--conv", "Conv3d"]
    resumed = ckpt.exists()
    status = {"run_id": run["run_id"], "dataset": run["dataset"], "seed": run["seed"],
              "dimension": run["dimension"], "model": "resnet50", "epochs": EPOCHS,
              "batch_size": run["batch_size"], "commit": UPSTREAM_COMMIT,
              "gpu_index": gpu_index, "resumed_from_checkpoint": resumed,
              "status": "running",
              "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
              "verified": False}
    if "prior_attempts" in run:
        status["prior_attempts"] = run["prior_attempts"]
    with LAUNCHER_LOCK:
        write_json(rd / "status.json", status)
        write_json(rd / "command.json", cmd)
    ensure_dataset_downloaded(run)
    env = child_environment(gpu_index, run["seed"], deadline)
    with open(rd / "train.log", "a", encoding="utf-8") as lf:
        lf.write("CMD " + " ".join(cmd) + "\n")
        lf.write("GPU_INDEX %d RESUMED %s DEADLINE %.0f\n" % (gpu_index, resumed, deadline))
        lf.flush()
        code, timed_out = run_process(cmd, cwd=str(BASE / "work"), env=env, log=lf,
                                      deadline=deadline, poll_seconds=30)
    status["ended_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    status["exit_code"] = code
    status["timed_out"] = timed_out
    status.update(read_progress(ckpt))
    checkpoint_state = None
    completed_epochs = 0
    if ckpt.exists():
        try:
            checkpoint_state = torch.load(ckpt, map_location="cpu", weights_only=False)
            completed_epochs = checkpoint_completed_epochs(checkpoint_state)
            status["completed_epochs"] = completed_epochs
        except (OSError, RuntimeError, ValueError, KeyError) as err:
            status["status"] = "failed"
            status["note"] = "checkpoint validation failed: " + str(err)
            write_json(rd / "status.json", status)
            return status
    # Paused cleanly at an epoch boundary: the checkpoint stays, the next session resumes.
    # This also catches the training patch's intentional code-0 return before the deadline.
    if ckpt.exists() and is_resumable_pause(code, timed_out, completed_epochs, EPOCHS):
        status["status"] = "paused"
        status["note"] = ("session budget reached; atomic epoch checkpoint saved - "
                          "the next session resumes from completed_epochs")
        write_json(rd / "status.json", status)
        print(run["run_id"], "PAUSED at epoch", status.get("completed_epochs"), "- resumable")
        return status
    if code != 0:
        status["status"] = "failed"
        status["note"] = "non-zero exit; see train.log before rerunning"
        write_json(rd / "status.json", status)
        print(run["run_id"], "FAILED, exit", code, "- rerun the loop cell to resume")
        return status

    if checkpoint_state is None:
        status["status"] = "failed"
        status["note"] = "training exited without a checkpoint"
        write_json(rd / "status.json", status)
        return status
    resolved = Path(checkpoint_state["output_root_resolved"]).resolve()
    expected_output_root = (BASE / "work" / "output" / run["run_id"]).resolve()
    try:
        resolved.relative_to(expected_output_root)
    except ValueError:
        status["status"] = "failed"
        status["note"] = "checkpoint output path is outside this run's isolated work directory"
        write_json(rd / "status.json", status)
        return status
    del checkpoint_state
    try:
        # Independent re-check: recompute every split from the saved prediction CSVs
        # with the official evaluator and require agreement with the final log line.
        checked = verify_artifacts(run, resolved, completed_epochs, EPOCHS)
    except ValueError as err:
        status["status"] = "failed"
        status["note"] = "artifact verification refused this run: " + str(err)
        write_json(rd / "status.json", status)
        print(run["run_id"], "NOT VERIFIED:", err)
        return status
    status.update(status="completed", test_auc=checked["test_auc"], test_acc=checked["test_acc"],
                  verified=True, verification_method=checked["verification_method"],
                  artifact_sha256=checked["artifact_sha256"])
    for item in resolved.iterdir():
        if item.is_dir():
            shutil.copytree(item, rd / item.name, dirs_exist_ok=True)
        elif KEEP_BEST_MODEL or item.name != "best_model.pth":
            shutil.copy2(item, rd / item.name)
    # Make completion durable first. An orphaned checkpoint is harmless; a deleted
    # checkpoint with a stale running status would force a complete retrain.
    write_json(rd / "status.json", status)
    if status.get("verified") and ckpt.exists():
        ckpt.unlink()  # resume checkpoint no longer needed once a run is verified
        progress = Path(str(ckpt) + ".progress.json")
        if progress.exists():
            progress.unlink()
    if resolved.exists():
        shutil.rmtree(resolved)
    print(run["run_id"], status["status"], "verified:", status["verified"])
    return status


## 8. Process this session

In [ ]:
import time as _time
import torch as _torch
import threading as _threading
from concurrent.futures import ThreadPoolExecutor

# One official run per physical GPU, run as independent processes with
# CUDA_VISIBLE_DEVICES isolation. This is deliberately NOT DataParallel / DDP:
# splitting a single model across GPUs would change the effective batch size and
# BatchNorm statistics and would no longer reproduce the published baselines.

if not _torch.cuda.is_available():
    raise SystemExit("CUDA is False. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all.")

GPU_COUNT = _torch.cuda.device_count()
WORKERS = max(1, min(PARALLEL_RUNS, GPU_COUNT))

print("GPUs visible:", GPU_COUNT, [(_torch.cuda.get_device_name(i)) for i in range(GPU_COUNT)])
print("workers (one run per GPU):", WORKERS, "| PARALLEL_RUNS =", PARALLEL_RUNS)
print("budget: MAX_SESSION_MINUTES =", MAX_SESSION_MINUTES, "| MIN_RUN_MINUTES =", MIN_RUN_MINUTES)
print("STOP_ON_FAILURE =", STOP_ON_FAILURE, "| dimensions:", DIMENSIONS_THIS_SESSION,
      "| scope:", DATASETS_THIS_SESSION)
if EXCLUDED_RUN_IDS:
    print("excluded run_ids (owned by another session):", EXCLUDED_RUN_IDS)

def minutes_left():
    return MAX_SESSION_MINUTES - (_time.time() - SESSION_START) / 60.0

_CLAIMED = set()
_CLAIM_LOCK = _threading.Lock()

def claim_next():
    """Reserve the next eligible run_id atomically, so no GPU ever races another."""
    with _CLAIM_LOCK:
        for run in PLAN_RUNS:
            if run["dimension"] not in DIMENSIONS_THIS_SESSION:
                continue
            if run["run_id"] in _CLAIMED or run["run_id"] in EXCLUDED_RUN_IDS:
                continue
            if run["dataset"] not in DATASETS_THIS_SESSION:
                continue
            if is_done(run):
                continue
            _CLAIMED.add(run["run_id"])
            return run
        return None

def worker(gpu_index):
    finished = []
    while minutes_left() > MIN_RUN_MINUTES:
        run = claim_next()
        if run is None:
            print(f"[gpu {gpu_index}] no eligible runs remain in scope")
            break
        print(f"=== START {run['run_id']} on gpu {gpu_index} ({minutes_left():.0f} min left) ===")
        status = launch(run, gpu_index=gpu_index)
        finished.append(status)
        print(f"=== END {run['run_id']}: status={status.get('status')} "
              f"verified={status.get('verified')} epochs={status.get('completed_epochs')} ===")
        if status.get("status") == "paused":
            print(f"[gpu {gpu_index}] session budget reached; stopping this worker")
            break
        if status.get("status") != "completed" or not status.get("verified"):
            if STOP_ON_FAILURE:
                print(f"[gpu {gpu_index}] stopping after a non-verified run; fix, then Run all to resume")
                break
    else:
        print(f"[gpu {gpu_index}] stopping: {minutes_left():.0f} min of budget remain "
              f"(< MIN_RUN_MINUTES={MIN_RUN_MINUTES})")
    return finished

results = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for part in pool.map(worker, range(WORKERS)):
        results.extend(part)

print(json.dumps([{"run_id": s["run_id"], "status": s["status"],
                   "completed_epochs": s.get("completed_epochs"),
                   "test_auc": s.get("test_auc"), "test_acc": s.get("test_acc"),
                   "verified": s["verified"]} for s in results], indent=2))
print(f"session used {minutes_left()*-1 + MAX_SESSION_MINUTES:.0f} of {MAX_SESSION_MINUTES} minutes")
if PLATFORM == "kaggle":
    print("Kaggle committed-version run finished; /kaggle/working will persist automatically.")
else:
    print("Drive-backed checkpoints and status files are already persistent.")


## 9. Progress summary

In [ ]:
rows = []
for run in PLAN_RUNS:
    st = read_status(run) or {}
    verified = st.get("status") == "completed" and st.get("verified") is True
    rows.append({"run_id": run["run_id"], "dimension": run["dimension"],
                 "dataset": run["dataset"], "seed": run["seed"],
                 "status": st.get("status", "planned"),
                 "auc": st.get("test_auc") if verified else None,
                 "acc": st.get("test_acc") if verified else None,
                 "verified": verified, "reported_auc": run["reported_auc"],
                 "reported_acc": run["reported_acc"]})
done = [r for r in rows if r["status"] == "completed" and r["verified"]]
print(f"verified official runs: {len(done)}/90")
import pandas as pd
run_table = pd.DataFrame(rows)

summary_rows = []
for (dimension, dataset), planned_group in run_table.groupby(["dimension", "dataset"], sort=False):
    group = planned_group[planned_group["verified"]]
    reference_auc = float(planned_group["reported_auc"].iloc[0])
    reference_acc = float(planned_group["reported_acc"].iloc[0])
    summary = {"dimension": dimension, "dataset": dataset,
               "verified_runs": len(group), "expected_runs": 5,
               "reported_auc": reference_auc, "reported_acc": reference_acc,
               "mean_auc": None, "std_auc": None, "auc_difference": None,
               "auc_loss": None, "auc_loss_under_1pct": None,
               "mean_acc": None, "std_acc": None, "acc_difference": None,
               "acc_loss": None, "acc_loss_under_1pct": None,
               "all_verified": len(group) == 5}
    if len(group) == 5:
        mean_auc, mean_acc = float(group["auc"].mean()), float(group["acc"].mean())
        auc_loss, acc_loss = max(0.0, reference_auc - mean_auc), max(0.0, reference_acc - mean_acc)
        summary.update(mean_auc=mean_auc, std_auc=float(group["auc"].std(ddof=1)),
                       auc_difference=mean_auc - reference_auc, auc_loss=auc_loss,
                       auc_loss_under_1pct=auc_loss < 0.01,
                       mean_acc=mean_acc, std_acc=float(group["acc"].std(ddof=1)),
                       acc_difference=mean_acc - reference_acc, acc_loss=acc_loss,
                       acc_loss_under_1pct=acc_loss < 0.01)
    summary_rows.append(summary)

summary_table = pd.DataFrame(summary_rows)
reports = BASE / "reports"
reports.mkdir(parents=True, exist_ok=True)
for table, filename in ((run_table, "run-results.csv"),
                        (summary_table, "dataset-summary.csv")):
    target = reports / filename
    temporary = target.with_suffix(target.suffix + ".tmp")
    table.to_csv(temporary, index=False)
    os.replace(temporary, target)
print("wrote verified progress tables to", reports)
display(summary_table)


## Integrity checklist (every run)

- Same official protocol: ResNet-50, 100 epochs, official data pipeline and evaluator.
- Seed recorded in `status.json`; checkpoint path fixed per run id.
- Metrics come only from the official evaluator output and final log.
- `verified` is set only when the evaluator CSV exists and matches the parsed log.
- Failed runs stay in `runs/<run_id>/` with their log; they are never deleted or hidden.
